# EXP-2026-006 / Q5-C — 공유 실패 핵심 (quest52)

**상태: DESIGN / RESULT NOT RUN.** 아래 셀을 실행하기 전까지 이 문서의 어떤 숫자도 결과가 아니다.

- **ANALYSIS ONLY / NO TRAINING** — 학습하지 않는다. 저장된 예측만 읽는다. GPU가 필요 없다.
- 묻는 것: Q5-A가 남긴 미해석 사실 — 서로 다른 계보인 V10과 V9_BASE가 **S beat의 절반 가까이를 동시에 틀린다**. 그런데 두 모델의 worst **환자**는 거의 안 겹친다(전 쌍 최소 overlap 0.333). 환자 단위로는 안 지속되는데 beat 단위로는 지속되는 이 물건이 무엇인가?
- **43.6%는 설명 대상이 아니다.** 그 수치는 임계값 기반(Q5-A가 이미 강등한 정의)이고, 두 모델이 각자 나빠서 우연히 겹치는 몫만 **29.1%(474박)** 다. 이 실험이 세는 것은 **우연 초과분**이다.
- 난이도는 **record 안에서만** 정의한다 — record를 가로질러 beat를 비교하면 Q5-A가 1위로 측정한 환자 효과가 그대로 섞여 들어온다. 그래서 우연 기준선이 `0.5**4 = 0.0625`로 **계산 없이** 확정된다.
- 설명은 **Q5-A가 이미 등록한 블록**(`B_ATRIAL`·`B_RR`·`B_QUALITY`)으로만 시도한다. **새 특징을 만들지 않는다.** `B_SUBTYPE`은 영구 종결(EXP-2026-005), `B_PATIENT`는 순환이라 제외.
- 여기서 말할 수 있는 것은 `failure-associated factor`(**실패 연관 요인**)까지다. `원인`은 요인 하나만 바꾸는 개입과 음성대조군으로만 검증한다.
- **residual CNN 경로는 closed**이며 재개하거나 변형을 제안하지 않는다. **INCART rescue run도 하지 않는다.**
- **어떤 개입도 여기서 구현하지 않는다.** D-A가 나와도 Q5-B는 별도 승인 대상이다.

spec: `experiments/specs/EXP-2026-006-q5c-shared-error-core.md`

## 사전등록 판정

| 분기 | 조건 | 뜻 |
|---|---|---|
| `NO_SHARED_CORE` (D-C) | 초과분 < 1.25배 또는 CI 하한 ≤ 1.0 | "공유 핵심"이란 틀이 틀렸다. 43.6%는 산수였다 |
| `SHARED_CORE_UNSTRUCTURED` (D-B) | 초과분은 실재하나 등록 블록이 환자 밖에서 판별 못함 | 핵심은 있으나 **측정한 무엇으로도 안 보인다** → 새 모델이 아니라 새 측정 |
| `SHARED_CORE_STRUCTURED` (D-A) | 판별 + 셔플 대조군 통과 | 후보 요인을 **지목**한다(개입 승인은 아니다) |

## mode 실행 순서 (정확히 하나만 활성)

| 순서 | mode | 하는 일 |
|---|---|---|
| ① | — | 셀 2: repo 준비 + Q5-A / Q5-C 회귀 테스트 |
| ② | — | 셀 3: Drive mount + 입력 경로 |
| ③ | `ANALYZE` | 셀 4: 공유 핵심 측정 + 사전등록 판정 |
| ④ | `REPORT` | 셀 5: 저장 bundle만 다시 표시 (재계산 없음) |

## 입력

Q5-A와 **완전히 동일**하다 — 동결 cohort(`mamba_data.npz`), Q5-A INVENTORY가 만든 `baseline_freeze.json`(V10 / V10_BASE / V9 / V9_BASE), baseline 패키지. 새 데이터를 쓰지 않는다.


In [ ]:
# ── 실행 설정 (정확히 하나의 mode) ────────────────────────────────────────────
VALID_MODES = ("DESIGN", "ANALYZE", "REPORT")
MODE = "DESIGN"          # DESIGN -> ANALYZE -> REPORT
assert MODE in VALID_MODES, f"MODE must be one of {VALID_MODES}"

BRANCH = "main"          # 이 notebook이 쓸 repo 브랜치
NEED_Q5C = 2             # 이 notebook이 요구하는 최소 q5c 모듈 버전
NEED_Q5A = 8             # Q5-A(인수본) 위에서만 돈다

REPORT_RUN = ""          # REPORT mode에서 읽을 run 폴더 (비우면 최신)
print("mode:", MODE)


In [ ]:
# ── 셀 2: repo 준비 + 회귀 테스트 ─────────────────────────────────────────────
import os, subprocess, sys

REPO = "/content/my-github-test"
os.chdir("/content")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone",
                    "https://github.com/ehdbddl06001-ui/my-github-test.git"],
                   check=True)
os.chdir(REPO)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

for _m in [m for m in list(sys.modules)
           if m.startswith(("q5a_", "q5b0_", "q5c_", "q4o_", "q4p_", "q4q_"))]:
    sys.modules.pop(_m, None)
sys.path.insert(0, os.path.join(REPO, "mit-bih"))

import q5a_patient_failure_atlas as QA
import q5c_shared_error_core as QC

assert QA.MODULE_VERSION >= NEED_Q5A, (
    f"stale module: q5a v{QA.MODULE_VERSION} < v{NEED_Q5A}")
assert QC.MODULE_VERSION >= NEED_Q5C, (
    f"stale module: q5c v{QC.MODULE_VERSION} < v{NEED_Q5C}. BRANCH 확인 후 "
    "Runtime > Restart runtime")
print("q5a v%d  <- %s" % (QA.MODULE_VERSION, QA.__file__))
print("q5c v%d  <- %s" % (QC.MODULE_VERSION, QC.__file__))

for suite in ("test_q5a_patient_failure_atlas", "test_q5b0_subtype_key_recovery",
              "test_q5c_shared_error_core"):
    r = subprocess.run([sys.executable, f"mit-bih/{suite}.py"],
                       capture_output=True, text=True)
    tail = [l for l in r.stdout.splitlines() if l.startswith("passed ")]
    print(f"{suite}: {tail[-1] if tail else 'NO RESULT'}"
          + ("" if r.returncode == 0 else "   <-- FAILED"))
    assert r.returncode == 0, f"{suite} failed — 여기서 멈춘다"


In [ ]:
# ── 셀 3: Drive mount + 입력 경로 (Q5-A와 동일한 입력) ──────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive"
SOURCE = f"{ROOT}/mitbih/mamba_data.npz"
RUNS = f"{ROOT}/MedKOS/ecg-model/runs"
PKGS = f"{ROOT}/mitbih/baseline_pkgs"

print(("OK   " if os.path.exists(SOURCE) else "MISS "), SOURCE)
os.makedirs(RUNS, exist_ok=True)


In [ ]:
# ── 셀 4: ANALYZE — 공유 핵심 측정 + 사전등록 판정 ──────────────────────────
import glob, json, time
import numpy as np
assert MODE == "ANALYZE", f"이 셀은 ANALYZE 전용 (지금 {MODE})"
assert QC.MODULE_VERSION >= NEED_Q5C, "stale module — 셀 2를 다시 실행"

log = QA.RunLog()
cohort, audit = QA.load_atlas_source(SOURCE, log=log)
QA.rr_from_samples(cohort)

# baseline은 Q5-A가 동결한 그대로 재사용한다
inv_dirs = sorted(glob.glob(f"{RUNS}/*_EXP-2026-004_q5a_inventory"))
assert inv_dirs, "Q5-A INVENTORY를 먼저 실행한다"
inv = json.load(open(f"{inv_dirs[-1]}/source_inventory.json", encoding="utf-8"))
freeze = json.load(open(f"{inv_dirs[-1]}/baseline_freeze.json", encoding="utf-8"))
assert freeze["status"].startswith("FROZEN"), freeze.get("reasons")
print("freeze:", freeze["status"], sorted(freeze.get("selected", {})))

source_index = QA.load_frozen_source_index(SOURCE, log=log)
models_raw = {}
for label, sel in freeze.get("selected", {}).items():
    if not sel.get("beat_level_ready"):
        continue
    models_raw[label] = QA.load_model_predictions(
        sel["run_dir"], label, source_index=source_index,
        arm=sel.get("model_name"), log=log)
assert models_raw, "beat-level 산출물을 하나도 읽지 못했다"

# Q5-A와 동일한 정렬·cohort 제한 (모든 모델이 공통으로 덮는 DS2 record)
split = QA.cohort_split(cohort)
key_index = {str(k): i for i, k in enumerate(cohort.key)}
covered = set(split["ds2"])
aligned = {}
for lab, m in models_raw.items():
    a = QA.match_beat_keys(cohort, m, strict=False)
    assert a["pass"], (lab, a["fail_reasons"])
    covered &= set(int(r) for r in a["model_record_scope"])
    full = np.full(cohort.n, np.nan)
    pos = np.array([key_index[str(k)] for k in m.key], int)
    full[pos] = m.score
    aligned[lab] = full
ds2 = sorted(covered)
excluded = sorted(set(split["ds2"]) - covered)
rows = np.sort(cohort.rows_of(ds2))
print(f"분석 cohort: {len(ds2)} record · 제외 {excluded}")

models = {lab: type("M", (), {"score": v[rows]})() for lab, v in aligned.items()}

STAMP = time.strftime("%Y%m%dT%H%M")
OUT = os.path.join(RUNS, QC.run_dir_name(STAMP))
res = QC.run_core_analysis(cohort, models, rows, OUT, n_boot=QC.NB_BOOT,
                           provenance={"atlas_source": audit,
                                       "ds2_analysis": ds2,
                                       "ds2_excluded": excluded,
                                       "notebook": "quest52_q5c_shared_error_core"},
                           log=log)

ex = res["co_error"]
print()
print("status:", res["status"], "| training_performed:", res["training_performed"])
print(f"모델 {res['models']}")
print(f"우연 기준선 {ex['chance']:.4f} (= 0.5^{ex['n_model']})")
print(f"실측 공유율(record 평균) {ex['observed']:.4f}  ->  **{ex['excess']:.2f}배**"
      f"  [{ex['ci_low']:.2f}, {ex['ci_high']:.2f}]  (환자 bootstrap)")
print(f"  (참고) beat 통합값 {ex['observed_micro']:.4f} -> {ex['excess_micro']:.2f}배"
      "  — 점추정과 CI는 같은 추정량(record 평균)이어야 한다")
print(f"대상 S beat {res['n_s_beat']} · record {res['n_record']}")
c = res["concentration"]
print(f"집중도: 핵심 절반을 {c['records_for_50pct']}개 record, 80%를 "
      f"{c['records_for_80pct']}개가 차지 ({c['spread']})")
j = (res.get("explain") or {}).get("joint") or {}
print(f"등록 블록의 환자 밖 판별: AUROC {j.get('auroc_aug')} · "
      f"Δ {j.get('delta_logloss')} [{j.get('ci_low')}, {j.get('ci_high')}]")
print("셔플 대조군:", {k: res["shuffle_control"].get(k) for k in
                      ("applicable", "real_delta", "shuffled_mean", "pass")})
print()
print("판정:", res["decision"]["branch"], f"({res['decision']['rule']})")
print("근거:", res["decision"]["reason"])
print("다음:", res["decision"]["next_step"])
print("bundle:", OUT)


In [ ]:
# ── 셀 5: REPORT — 저장 bundle만 다시 표시 (재계산 없음) ────────────────────
assert MODE == "REPORT", f"이 셀은 REPORT 전용 (지금 {MODE})"

run = REPORT_RUN or sorted(
    [os.path.join(RUNS, d) for d in os.listdir(RUNS) if "EXP-2026-006" in d])[-1]
rep = QC.report_bundle(run)
print("run:", run, "| status:", rep.get("status"))
print()
print(rep.get("summary", "(no summary)"))
